# Step 2 — SimCLR Self-Supervised Pretraining

**Project:** Continual Self-Supervised Pretraining for Industrial Anomaly Localization

**Author:** Rafay Saif, Umar Aziz

**Goal of this notebook:**
1. Implement SimCLR's NT-Xent contrastive loss from scratch (not via a library) --
   you need to be able to derive and explain this for your viva.
2. Build an augmentation pipeline tuned for industrial imagery (reduced color
   jitter -- see proposal Section 3.3).
3. Pretrain a ResNet-18 encoder on `train/good` images only, with NO labels.
4. Checkpoint every epoch to Drive.
5. Sanity-check the trained encoder before trusting it (this matters as much
   as the mask-overlay check in Notebook 1 -- a silently broken contrastive
   loss often still trains "successfully" with decreasing loss while learning
   nothing useful, e.g. via a representation collapse).

**Requires GPU runtime.** If you haven't switched yet: Runtime > Change
runtime type > GPU, then re-run cells 2.1-2.2 below to remount Drive.

## 2.1 — Remount Drive, confirm GPU, restore paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), "No GPU detected -- switch Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

device = torch.device("cuda")

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/anomaly_project'
DATA_ROOT = f'{PROJECT_ROOT}/data/mvtec'
CHECKPOINT_ROOT = f'{PROJECT_ROOT}/checkpoints'
RESULTS_ROOT = f'{PROJECT_ROOT}/results'
LOG_ROOT = f'{PROJECT_ROOT}/logs'

for d in [DATA_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT, LOG_ROOT]:
    os.makedirs(d, exist_ok=True)

CATEGORY = "bottle"  # validate on this first; cable/leather follow identically
category_path = f'{DATA_ROOT}/{CATEGORY}'
assert os.path.exists(f'{category_path}/train/good'), (
    f"'{CATEGORY}' not found at {category_path} -- run Notebook 1 first for this category."
)
print(f"Using category: {CATEGORY} ({len(os.listdir(f'{category_path}/train/good'))} train images)")

## 2.2 — Re-declare `MVTecDataset` and checkpoint utilities

GPU runtime switch wiped the Python session, so these need to be re-defined
(identical to Notebook 1 -- copy-pasted here so this notebook is self-contained
and can be run independently).

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from pathlib import Path
import time
import json


class MVTecDataset(Dataset):
    """MVTec AD dataset loader. See Notebook 1 for full documentation."""

    def __init__(self, root, split="train", transform=None, mask_transform=None, two_crop=False):
        self.root = Path(root)
        self.split = split
        self.transform = transform
        self.mask_transform = mask_transform
        self.two_crop = two_crop
        self.samples = self._build_index()

    def _build_index(self):
        samples = []
        if self.split == "train":
            good_dir = self.root / "train" / "good"
            for img_path in sorted(good_dir.glob("*.png")):
                samples.append({"image_path": img_path, "label": 0, "mask_path": None})
        elif self.split == "test":
            test_dir = self.root / "test"
            for defect_dir in sorted(test_dir.iterdir()):
                if not defect_dir.is_dir():
                    continue
                defect_type = defect_dir.name
                is_anomalous = defect_type != "good"
                for img_path in sorted(defect_dir.glob("*.png")):
                    mask_path = None
                    if is_anomalous:
                        mask_path = self.root / "ground_truth" / defect_type / f"{img_path.stem}_mask.png"
                    samples.append({"image_path": img_path, "label": int(is_anomalous), "mask_path": mask_path})
        else:
            raise ValueError(f"split must be 'train' or 'test', got '{self.split}'")
        return samples

    def __len__(self):
        return len(self.samples)

    def _load_image(self, path):
        return Image.open(path).convert("RGB")

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = self._load_image(sample["image_path"])

        if self.split == "train":
            if self.two_crop:
                view1 = self.transform(img) if self.transform else img
                view2 = self.transform(img) if self.transform else img
                return view1, view2
            else:
                return self.transform(img) if self.transform else img
        else:
            img_t = self.transform(img) if self.transform else img
            if sample["mask_path"] is not None:
                mask = Image.open(sample["mask_path"]).convert("L")
                mask_t = self.mask_transform(mask) if self.mask_transform else mask
                mask_t = torch.from_numpy((np.array(mask_t) > 0).astype(np.float32))
            else:
                h, w = img_t.shape[-2:] if isinstance(img_t, torch.Tensor) else img.size[::-1]
                mask_t = torch.zeros((h, w), dtype=torch.float32)
            return {"image": img_t, "label": sample["label"], "mask": mask_t,
                    "image_path": str(sample["image_path"])}


def save_checkpoint(state: dict, category: str, strategy: str, tag: str = "latest"):
    out_dir = f"{CHECKPOINT_ROOT}/{category}"
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{strategy}_{tag}.pt"
    state["_saved_at"] = time.time()
    torch.save(state, path)
    return path


def load_checkpoint(category: str, strategy: str, tag: str = "latest", map_location="cpu"):
    path = f"{CHECKPOINT_ROOT}/{category}/{strategy}_{tag}.pt"
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location)


def save_result(result: dict, category: str, strategy: str):
    path = f"{RESULTS_ROOT}/{category}_{strategy}.json"
    with open(path, "w") as f:
        json.dump(result, f, indent=2, default=str)
    print(f"Result saved: {path}")

print("Dataset class and checkpoint utilities ready.")

## 2.3 — Augmentation pipeline (industrial-imagery variant)

**The core idea behind SimCLR's augmentations:** the model is forced to produce
the *same* embedding for two differently-augmented views of the *same* image,
while pushing embeddings of *different* images apart. The choice of
augmentations defines what invariances the encoder learns -- e.g. random crop
teaches scale/translation invariance, color jitter teaches lighting invariance.

**Why we deviate from the original SimCLR recipe here:** the original paper
(Chen et al., 2020) tunes augmentations for natural images (ImageNet), where
color is rarely the defining feature of object identity. In MVTec categories
like Leather and Pill, however, **color deviation IS a defect signal** -- a
stain or discoloration is exactly the kind of anomaly we want the downstream
model to be sensitive to. If the SSL pretraining stage is trained to treat
color shifts as nuisance variation to be invariant to, it may suppress the
very signal anomaly detection depends on.

**Our adjustment:** color jitter strength reduced from the SimCLR-standard 0.5
to 0.2, and `RandomGrayscale` (which can be even more aggressive at destroying
color information) is disabled entirely. This is itself Ablation A2 in your
proposal -- this notebook trains the `jitter=0.2` variant; re-running with
`COLOR_JITTER_STRENGTH = 0.5` or `0.1` produces the other ablation arms.

In [ ]:
import torchvision.transforms as T

IMG_SIZE = 224  # standard ResNet input size

# torchvision's ColorJitter hue parameter rotates color around the full
# 360-degree hue wheel, not a perceptually uniform scale. A hue value of 0.1
# allows up to a 36-degree rotation, which can be large for desaturated
# brown/tan tones specifically -- brown sits in a perceptually compressed
# region of hue space, so a numerically small rotation can visibly shift it
# toward olive or red. For texture categories (Leather, Carpet, Tile, Wood,
# Grid in the full MVTec set), grain pattern carries identity more than
# overall color does, and a defect can itself be a color shift (e.g.
# staining), so hue jitter is disabled entirely for these categories rather
# than tuned down further.
#
# Texture categories also use a smaller minimum crop area. With
# scale=(0.5, 1.0), the smallest crop still covers half the image -- for a
# near-stationary texture this still captures roughly the same grain
# statistics as the full image, so two crops from DIFFERENT swatches can end
# up looking about as similar as two crops from the SAME swatch, which
# undermines the contrastive objective (negatives are not actually very
# different from positives). A lower scale bound of 0.2 forces smaller, more
# spatially localized crops so the task depends on local texture detail
# rather than whole-swatch color/lighting that different swatches can share
# by chance.

TEXTURE_CATEGORIES = {"leather", "carpet", "tile", "wood", "grid"}

COLOR_JITTER_STRENGTH = 0.2  # brightness/contrast/saturation -- unchanged for all categories
HUE_JITTER_STRENGTH = 0.0 if CATEGORY in TEXTURE_CATEGORIES else COLOR_JITTER_STRENGTH * 0.5
CROP_SCALE_LOWER_BOUND = 0.2 if CATEGORY in TEXTURE_CATEGORIES else 0.5

if CATEGORY in TEXTURE_CATEGORIES:
    print(f"'{CATEGORY}' is a texture category -- hue jitter disabled, "
          f"crop scale lower bound reduced to {CROP_SCALE_LOWER_BOUND}.")

# ImageNet normalization stats -- used even though ImageNet weights are not
# loaded here, because they are a reasonable, well-tested default for
# natural-image statistics and keep numerics in a stable range for the conv stem.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

simclr_train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(CROP_SCALE_LOWER_BOUND, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(
        brightness=COLOR_JITTER_STRENGTH,
        contrast=COLOR_JITTER_STRENGTH,
        saturation=COLOR_JITTER_STRENGTH,
        hue=HUE_JITTER_STRENGTH,
    ),
    # RandomGrayscale is intentionally omitted -- color information matters
    # for anomaly signal in this domain (see rationale above).
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Eval-time transform: deterministic, no augmentation -- used later for
# building the PatchCore memory bank and for test-time scoring.
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Augmentation pipelines defined.")
print(f"  Category: {CATEGORY}")
print(f"  Color jitter (brightness/contrast/saturation): {COLOR_JITTER_STRENGTH}")
print(f"  Hue jitter: {HUE_JITTER_STRENGTH}")
print("  RandomGrayscale: disabled")

## 2.4 — Visual check: do the two augmented views still look like the same object?


In [ ]:
import matplotlib.pyplot as plt

def denormalize(tensor):
    """Reverse ImageNet normalization for visualization."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

preview_ds = MVTecDataset(category_path, split="train", transform=simclr_train_transform, two_crop=True)

fig, axes = plt.subplots(3, 2, figsize=(6, 9))
for row in range(3):
    view1, view2 = preview_ds[row]
    axes[row, 0].imshow(denormalize(view1).permute(1, 2, 0))
    axes[row, 0].set_title(f"Sample {row} -- View A")
    axes[row, 0].axis("off")
    axes[row, 1].imshow(denormalize(view2).permute(1, 2, 0))
    axes[row, 1].set_title(f"Sample {row} -- View B")
    axes[row, 1].axis("off")
plt.tight_layout()
plt.show()

print("CHECK: each row should clearly show the same bottle from a similar")
print("viewpoint, with noticeable but not extreme differences in crop/color.")
print("If the two views look like completely different objects, augmentation")
print("is too aggressive -- reduce RandomResizedCrop's scale lower bound.")

## 2.5 — The encoder: ResNet-18 backbone with a projection head

**Architecture intuition:** SimCLR uses two components:
1. **Backbone encoder** `f(.)` -- a standard CNN (ResNet-18 here) that produces
   a feature vector. This is the part we actually want to keep after
   pretraining -- it becomes the feature extractor for PatchCore later.
2. **Projection head** `g(.)` -- a small MLP that maps the encoder's output
   into a lower-dimensional space where the contrastive loss is computed.

In [ ]:
import torch.nn as nn
import torchvision.models as models


class SimCLREncoder(nn.Module):
    """
    ResNet-18 backbone (no ImageNet weights -- random init, per proposal
    Strategy C) + a 2-layer MLP projection head.

    forward() returns BOTH:
      - h: the backbone feature (512-dim for ResNet-18) -- this is what gets
           saved and reused for PatchCore after pretraining.
      - z: the projected embedding (128-dim) -- this is what the NT-Xent
           loss operates on during pretraining, then discarded.
    """

    def __init__(self, projection_dim=128):
        super().__init__()
        backbone = models.resnet18(weights=None)  # weights=None -- random init, NOT ImageNet
        backbone_dim = backbone.fc.in_features  # 512 for ResNet-18
        backbone.fc = nn.Identity()  # strip the original 1000-way classifier head
        self.backbone = backbone
        self.backbone_dim = backbone_dim

        self.projection_head = nn.Sequential(
            nn.Linear(backbone_dim, backbone_dim),
            nn.ReLU(inplace=True),
            nn.Linear(backbone_dim, projection_dim),
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.projection_head(h)
        return h, z


encoder = SimCLREncoder(projection_dim=128).to(device)
n_params = sum(p.numel() for p in encoder.parameters())
print(f"SimCLREncoder created: {n_params:,} parameters")
print(f"Backbone feature dim: {encoder.backbone_dim}")

# Smoke test: confirm shapes are correct before training
_dummy = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(device)
_h, _z = encoder(_dummy)
print(f"Smoke test -- backbone output shape: {_h.shape}, projection output shape: {_z.shape}")
assert _h.shape == (4, 512) and _z.shape == (4, 128), "Unexpected output shape!"
print("Encoder smoke test PASSED.")

## 2.6 — The NT-Xent loss, derived from first principles

This is the mathematical core of SimCLR, and the part of this notebook you
should be able to fully explain in a viva.

**Setup.** Given a batch of $N$ images, we create $2N$ augmented views (two
per image: view A and view B). For a given image $i$, its two views $z_i^A$
and $z_i^B$ form a **positive pair** -- everything else in the batch (the
other $2N-2$ views) are **negative pairs**.

**Similarity.** We measure closeness in the projection space using cosine
similarity:
$$\text{sim}(u, v) = \frac{u^\top v}{\|u\|\|v\|}$$

**The loss for a single positive pair $(i, j)$** (NT-Xent = "Normalized
Temperature-scaled Cross Entropy"):
$$\ell(i, j) = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)}$$

**Reading this equation as a classification problem:** this is exactly the
softmax cross-entropy loss, where the "correct class" for view $z_i$ is its
positive partner $z_j$, and the "incorrect classes" are all $2N-2$ other views
in the batch. The numerator pulls the positive pair's similarity up; the
denominator (summed over ALL other views, positive included) normalizes this
into a probability and simultaneously pushes down similarity to every
negative.

**The temperature $\tau$.** This is a sharpening parameter. As $\tau \to 0$,
the softmax becomes closer to a hard argmax -- the loss penalizes the model
extremely harshly for any negative pair with even moderately high similarity.
As $\tau \to \infty$, all similarities get scaled toward 0 and the softmax
becomes closer to uniform, making the loss nearly insensitive to which
negative is most confusable. SimCLR's original paper found $\tau = 0.5$
worked well empirically; we use the same value here as a default.

**Total batch loss.** Averaged symmetrically over both directions of every
positive pair (i.e. $\ell(i,j)$ AND $\ell(j,i)$ are both included), summed over
all $N$ pairs, divided by $2N$.

In [ ]:
import torch.nn.functional as F


def nt_xent_loss(z_a, z_b, temperature=0.5):
    """
    NT-Xent contrastive loss (SimCLR), implemented explicitly rather than via
    a library, so every step matches the derivation above 1:1.

    Parameters
    ----------
    z_a, z_b : torch.Tensor, shape (N, D)
        Projection-head outputs for view A and view B respectively.
        z_a[i] and z_b[i] are the two augmented views of the SAME image i --
        i.e. they form a positive pair.
    temperature : float
        The tau in the derivation above.

    Returns
    -------
    torch.Tensor, scalar
        The mean NT-Xent loss over the batch.
    """
    N = z_a.shape[0]

    # Step 1: concatenate views A and B into one (2N, D) tensor.
    # Index i in [0, N) is view A of image i; index N+i is view B of image i.
    z = torch.cat([z_a, z_b], dim=0)  # (2N, D)

    # Step 2: L2-normalize so dot product = cosine similarity directly.
    z = F.normalize(z, dim=1)

    # Step 3: full pairwise similarity matrix, scaled by temperature.
    # sim_matrix[a, b] = sim(z[a], z[b]) / tau
    sim_matrix = torch.matmul(z, z.T) / temperature  # (2N, 2N)

    # Step 4: mask out self-similarity (k != i in the derivation's denominator) --
    # an embedding is always maximally similar to itself, which would trivially
    # dominate the denominator and isn't a meaningful negative.
    self_mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
    sim_matrix.masked_fill_(self_mask, float("-inf"))

    # Step 5: build the positive-pair index for every row.
    # Row i (0 <= i < N)       -> its positive partner is at index N + i.
    # Row N+i (0 <= i < N)     -> its positive partner is at index i.
    positive_indices = torch.cat([
        torch.arange(N, 2 * N, device=z.device),  # for rows [0, N): partner is N+i
        torch.arange(0, N, device=z.device),       # for rows [N, 2N): partner is i
    ])

    # Step 6: this IS exactly cross-entropy where the "label" for row i is its
    # positive_indices[i] -- matching the softmax-over-similarities reading of
    # the NT-Xent formula directly.
    loss = F.cross_entropy(sim_matrix, positive_indices)
    return loss


# Sanity test 1: identical embeddings for all positive pairs, distinct
# negatives -> loss should be small (near the theoretical minimum, not zero,
# since negatives still contribute small but nonzero terms to the denominator).
_z_a = F.normalize(torch.randn(8, 16), dim=1)
_z_b = _z_a.clone()  # perfect positive pairs: view B == view A exactly
_loss_easy = nt_xent_loss(_z_a, _z_b, temperature=0.5)
print(f"Sanity test 1 (perfect positive pairs): loss = {_loss_easy.item():.4f} (should be low)")

# Sanity test 2: random, unrelated embeddings for A and B -> loss should be
# noticeably higher, since there's no real signal connecting positive pairs.
_z_b_random = F.normalize(torch.randn(8, 16), dim=1)
_loss_hard = nt_xent_loss(_z_a, _z_b_random, temperature=0.5)
print(f"Sanity test 2 (random/unrelated pairs):  loss = {_loss_hard.item():.4f} (should be higher)")

assert _loss_easy.item() < _loss_hard.item(), "BUG: loss should be lower for perfectly matched pairs!"
print("\nNT-Xent sanity checks PASSED -- loss correctly rewards matched positive pairs.")

## 2.7 — Training loop with epoch-level checkpointing

**Hyperparameters chosen and why:**
- **Batch size 64** (not SimCLR's original 4096) -- the original paper used
  huge batches because more negatives per batch generally helps contrastive
  learning, but that requires distributed training across many GPUs. On a
  single Colab GPU, 64 is a practical ceiling for ResNet-18 at 224x224 without
  running out of VRAM. This is a real, documented limitation of small-scale
  SimCLR reproductions -- worth mentioning explicitly in your report's
  limitations section.
- **LARS or Adam?** The original paper uses LARS (Layer-wise Adaptive Rate
  Scaling), designed for huge batch sizes. At batch size 64, plain Adam is
  simpler and sufficiently stable -- using LARS here would be over-engineering
  relative to our batch size.
- **Epochs**: MVTec categories are small (209 train images for Bottle). We
  default to 100 epochs, which at this dataset size is roughly equivalent to
  ~330 gradient steps total at batch size 64 -- short by ImageNet-scale SSL
  standards, but appropriate given the dataset size.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64
NUM_EPOCHS = 300  # extended from the original 100-epoch schedule -- see rationale below
LEARNING_RATE = 3e-4
TEMPERATURE = 0.5
STRATEGY_NAME = "simclr"

# Bottle's 300-epoch SimCLR encoder outperformed both the 100-epoch version
# and the scratch baseline on PatchCore AUROC/PRO, while the original
# 100-epoch result had been roughly tied with scratch. This indicates the
# original schedule was undertrained rather than the small per-category
# dataset size being a hard ceiling, so the same extended schedule is applied
# here to all three categories for a fair comparison. A category-specific
# checkpoint tag keeps each 300-epoch run separate from its corresponding
# 100-epoch result so both can still be compared directly.

CHECKPOINT_TAG_300EP = {
    "bottle": "latest_300ep",
    "cable": "latest_300ep",
    "leather": "latest_v3_300ep",
}
CHECKPOINT_TAG = CHECKPOINT_TAG_300EP[CATEGORY]

train_ds = MVTecDataset(category_path, split="train", transform=simclr_train_transform, two_crop=True)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
    # drop_last=True avoids a final undersized batch, which would change the
    # number of negatives in NT-Xent's denominator inconsistently across batches.
)

print(f"Training on {len(train_ds)} images, batch size {BATCH_SIZE} "
      f"({len(train_loader)} batches/epoch), {NUM_EPOCHS} epochs.")
print(f"Checkpoint tag: '{CHECKPOINT_TAG}'")

In [ ]:
# Resume-from-checkpoint logic -- critical for surviving Colab disconnects.
# This checks Drive for a previous run BEFORE creating a fresh encoder, so
# re-running this cell after a disconnect continues training instead of
# restarting from epoch 0.

encoder = SimCLREncoder(projection_dim=128).to(device)
optimizer = torch.optim.Adam(encoder.parameters(), lr=LEARNING_RATE)
start_epoch = 0
loss_history = []

checkpoint = load_checkpoint(category=CATEGORY, strategy=STRATEGY_NAME, tag=CHECKPOINT_TAG, map_location=device)
if checkpoint is not None:
    encoder.load_state_dict(checkpoint["encoder_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    start_epoch = checkpoint["epoch"] + 1
    loss_history = checkpoint.get("loss_history", [])
    print(f"Resuming from epoch {start_epoch} (checkpoint had {len(loss_history)} recorded epochs)")
else:
    print("No checkpoint found -- starting fresh from epoch 0.")

In [ ]:
import time as _time

encoder.train()
training_start = _time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_losses = []
    epoch_start = _time.time()

    for view_a, view_b in train_loader:
        view_a = view_a.to(device, non_blocking=True)
        view_b = view_b.to(device, non_blocking=True)

        _, z_a = encoder(view_a)
        _, z_b = encoder(view_b)

        loss = nt_xent_loss(z_a, z_b, temperature=TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    mean_epoch_loss = sum(epoch_losses) / len(epoch_losses)
    loss_history.append(mean_epoch_loss)
    epoch_time = _time.time() - epoch_start

    if epoch % 5 == 0 or epoch == NUM_EPOCHS - 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | loss = {mean_epoch_loss:.4f} | {epoch_time:.1f}s/epoch")

    # Checkpoint EVERY epoch -- this is the non-negotiable habit established
    # in Notebook 1. Cheap to do, and the only thing standing between you and
    # losing an hour of GPU time to a Colab disconnect.
    save_checkpoint({
        "encoder_state": encoder.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": epoch,
        "loss_history": loss_history,
        "category": CATEGORY,
        "strategy": STRATEGY_NAME,
        "hyperparams": {
            "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
            "temperature": TEMPERATURE, "color_jitter": COLOR_JITTER_STRENGTH,
        },
    }, category=CATEGORY, strategy=STRATEGY_NAME, tag=CHECKPOINT_TAG)

total_time = _time.time() - training_start

## 2.8 — Sanity checks on the trained encoder

**Why this matters:** a contrastive loss can
"successfully" decrease over training while the encoder learns something
useless -- the most common failure is **representation collapse**, where the
encoder maps every image to the same (or nearly the same) point in feature
space. This trivially satisfies "positive pairs are similar" but destroys all
discriminative information, making the encoder worthless for anomaly
detection. A collapsed encoder's loss curve often still looks fine, so you
cannot detect this from the loss alone.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("NT-Xent loss")
plt.title(f"SimCLR pretraining loss -- {CATEGORY}")
plt.grid(alpha=0.3)
plt.show()

print(f"Loss: {loss_history[0]:.4f} (epoch 0) -> {loss_history[-1]:.4f} (epoch {len(loss_history)-1})")
print("CHECK: loss should decrease and roughly plateau. A loss that goes to")
print("near-zero almost immediately is suspicious -- possible collapse (see next cell).")

In [ ]:
# Collapse check: extract backbone features for a batch of DIFFERENT training
# images (no augmentation -- use eval_transform) and measure how spread out
# they are. If the encoder collapsed, all feature vectors will be nearly
# identical -- pairwise cosine similarity will be close to 1.0 for everything.

encoder.eval()
eval_ds = MVTecDataset(category_path, split="train", transform=eval_transform, two_crop=False)
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False)

all_features = []
with torch.no_grad():
    for batch in eval_loader:
        batch = batch.to(device)
        h, _ = encoder(batch)
        all_features.append(h.cpu())
all_features = torch.cat(all_features, dim=0)

normed = F.normalize(all_features, dim=1)
pairwise_sim = torch.matmul(normed, normed.T)

# Exclude the diagonal (self-similarity, trivially 1.0) before computing stats
off_diag_mask = ~torch.eye(pairwise_sim.shape[0], dtype=torch.bool)
off_diag_sims = pairwise_sim[off_diag_mask]

print(f"Pairwise cosine similarity across {len(all_features)} training images:")
print(f"  Mean:   {off_diag_sims.mean().item():.4f}")
print(f"  Std:    {off_diag_sims.std().item():.4f}")
print(f"  Min:    {off_diag_sims.min().item():.4f}")
print(f"  Max:    {off_diag_sims.max().item():.4f}")
print()

if off_diag_sims.mean().item() > 0.95 and off_diag_sims.std().item() < 0.02:
    print("WARNING: features look collapsed (near-identical for all images).")
    print("Possible causes: learning rate too high, too few epochs, or a bug")
    print("in the augmentation/loss pipeline. Do not proceed to PatchCore with")
    print("this checkpoint -- diagnose this first.")
else:
    print("Features show healthy variation across different images --")
    print("no evidence of collapse. Safe to proceed.")

In [ ]:
# Second sanity check: do visually similar images get more similar embeddings
# than visually different ones? We can't fully verify this without labels,
# but we CAN check that augmented views of the SAME image are more similar to
# each other than to a randomly chosen different image -- this is the most
# direct behavioral test of whether contrastive training actually worked.

check_ds = MVTecDataset(category_path, split="train", transform=simclr_train_transform, two_crop=True)

with torch.no_grad():
    img_a_view1, img_a_view2 = check_ds[0]
    img_b_view1, _ = check_ds[1]

    batch = torch.stack([img_a_view1, img_a_view2, img_b_view1]).to(device)
    h, _ = encoder(batch)
    h = F.normalize(h, dim=1)

    sim_same_image = (h[0] @ h[1]).item()      # two views of image A
    sim_different_image = (h[0] @ h[2]).item()  # image A vs image B

print(f"Similarity between two augmented views of the SAME image:      {sim_same_image:.4f}")
print(f"Similarity between two views of DIFFERENT images:               {sim_different_image:.4f}")
print()
if sim_same_image > sim_different_image:
    print("PASSED: same-image similarity is higher, as expected for a")
    print("correctly-trained contrastive encoder.")
else:
    print("WARNING: same-image similarity is NOT higher than different-image")
    print("similarity. This suggests training did not work as intended --")
    print("investigate before proceeding (check augmentation strength, learning")
    print("rate, or whether enough epochs have actually run).")

## Step 2 — Done. What you should have right now:

**Next steps:**
1. Re-run this notebook with `CATEGORY = "cable"` and `CATEGORY = "leather"` --
   only that one line changes, everything else is identical.
2. **Notebook 3**: build the PatchCore memory bank on top of these trained
   encoders, and compute the first real AUROC / PRO numbers -- this is where
   Phase 1 of your proposal (representation comparison) gets its first actual
   result.
3. Later: re-run this notebook with `STRATEGY_NAME = "scratch"` (skip training
   entirely, save the randomly-initialized encoder as-is) and with ImageNet
   pretrained weights (`weights="IMAGENET1K_V1"` in `models.resnet18(...)`) to
   get the other two Phase 1 baselines for comparison.